# 01. Plant Disease Dataset Exploration

Welcome to the **Plant Disease AI** initial dataset exploration notebook.

### Objectives:
1. **Dataset Structure Analysis**: How many total images and disease categories do we have?
2. **Class Balance Check**: Are the classes balanced or skewed across categories?
3. **Visual Inspection**: What do the leaf images and disease symptoms look like?
4. **Resolution & Image Dimensions**: What are the original image sizes and aspect ratios?
5. **Preprocessing Requirements**: What transformations, augmentations, and color normalizations are needed for CNN modeling?

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Set plot styles
plt.style.use('ggplot')
%matplotlib inline

## 1. Dataset Directory Setup & Scanning

In [ ]:
DATA_DIR = os.path.join('..', 'data')
print(f"[*] Base Data Directory: {os.path.abspath(DATA_DIR)}")

# Scan data directories
if os.path.exists(DATA_DIR):
    subdirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Found subdirectories: {subdirs}")
else:
    print("Data directory not found. Please place dataset under data/")

## 2. Image Distribution & Class Balance Analysis

In [ ]:
# Example helper function to parse dataset statistics
def analyze_dataset_distribution(root_dir):
    records = []
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(valid_exts):
                cls_name = os.path.basename(root)
                filepath = os.path.join(root, file)
                records.append({'filepath': filepath, 'class': cls_name})
                
    df = pd.DataFrame(records)
    return df

df_data = analyze_dataset_distribution(DATA_DIR)
print(f"Total Images Found: {len(df_data)}")
if not df_data.empty:
    print(df_data['class'].value_counts())
else:
    print("No images found yet in data/ directory.")

## 3. Visualizing Sample Leaf Images & Resolutions

In [ ]:
# Visualization function for inspecting image resolutions and samples
if not df_data.empty:
    sample_df = df_data.groupby('class').first().reset_index()
    fig, axes = plt.subplots(1, min(4, len(sample_df)), figsize=(15, 5))
    
    if len(sample_df) == 1:
        axes = [axes]
        
    for i, row in sample_df.head(4).iterrows():
        img = cv2.imread(row['filepath'])
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w, c = img.shape
            axes[i].imshow(img_rgb)
            axes[i].set_title(f"{row['class']}\n({w}x{h})")
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.show()

## 4. Preprocessing & Augmentation Strategy
- Resizing to standard input size: $128 \times 128$ or $224 \times 224$
- Color Normalization (ImageNet mean & std)
- Data Augmentation: Random rotations, horizontal flips, and brightness adjustments to combat overfitting.